# Predicting reaction time from EEG: an almost invisible signal

IODAA project · EEG Foundation Challenge 1 · releases R1 to R5 (99 subjects, 6,613 trials)

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch, butter, filtfilt
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

# data prepared beforehand (EEG windows, reaction times, subject ids, release of each trial)
X_win = np.load("data/X_fen_5rel.npy")
y = np.load("data/y_5rel.npy")
groups = np.load("data/groupes_5rel.npy", allow_pickle=True)
releases = np.load("data/releases_5rel.npy", allow_pickle=True)

L = []
for release in sorted(Path("data").glob("R*_mini_L100_bdf")):
    p = pd.read_csv(release / "participants.tsv", sep="\t")
    p["release"] = release.name
    L.append(p)
participants = pd.concat(L, ignore_index=True)

print(X_win.shape, y.shape, len(np.unique(groups)), "subjects")

## 1. Context: the dataset and the experiment

**EEG (electroencephalography).** Electrodes placed on the scalp measure tiny variations of electrical voltage (a few tens of microvolts) produced by neuronal activity. Here: 129 electrodes ("channels") and 100 samples per second (100 Hz). The signal is noisy: eye movements, muscles, poor contacts.

**The dataset: HBN-EEG** (Healthy Brain Network, Child Mind Institute). More than 3,000 children and teenagers aged 5 to 21, recorded during 6 tasks. The data are public and released in batches, the *releases*. I use the *mini* versions of **R1 to R5**: 20 subjects each, i.e. 100 subjects (99 after cleaning). A `participants.tsv` file gives each subject's age, sex and questionnaire scores.

**The competition: EEG Foundation Challenge (NeurIPS 2025).** Its first track asks to predict, from EEG, how fast a subject reacts.

**The experiment: Contrast Change Detection.**
1. Two striped discs flicker on screen, one tilted left, the other tilted right.
2. After a random delay, the contrast of one disc rises while the other's drops: this is the **stimulus**.
3. The subject presses a left or right button to indicate the disc whose contrast dominates: this is the **button press**.
4. Immediate feedback follows (a smiley or a sad face).

Each subject goes through several dozen trials. A trial's **reaction time** is the delay between the stimulus and the button press.

| Term | Meaning |
|---|---|
| Channel | One electrode, i.e. one signal over time. |
| Window | A 2 s chunk of EEG, an array `(129 channels, 200 samples)`. |
| RMSE | The model's average error, in seconds (lower is better). |
| Baseline | A very simple model used as a reference to beat. |
| `GroupKFold` | A validation scheme where trials from the same subject are never both in training and in test. |

In [ ]:
mean_rt = y.mean()

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.axhline(0, color="black", lw=1)
ax.axvspan(0.5, 2.5, ymin=0.1, ymax=0.9, color="orange", alpha=0.2)
ax.text(1.5, 0.62, "EEG window analysed (2 s)", ha="center", color="darkorange")
ax.annotate("", xy=(mean_rt, 0.3), xytext=(0, 0.3), arrowprops=dict(arrowstyle="<->"))
ax.text(mean_rt / 2, 0.36, f"reaction time ≈ {mean_rt:.1f} s", ha="center")
ax.plot([0], [0], "o", color="tab:red", ms=9)
ax.text(0, -0.15, "stimulus\n(the contrast changes)", ha="center", va="top")
ax.plot([mean_rt], [0], "o", color="tab:green", ms=9)
ax.text(mean_rt, -0.15, "button press\nby the subject", ha="center", va="top")
ax.set_xlim(-0.4, 3.0); ax.set_ylim(-0.9, 0.9); ax.axis("off")
plt.show()

## 2. The problem: predicting reaction speed from 2 s of EEG

- **Input**: a 2 s EEG window, starting 0.5 s after the stimulus and therefore covering the moment of the response.
- **Target**: the reaction time, in seconds.
- **Dataset**: 6,613 trials from 99 subjects, from the 5 releases.
- **Cleaning**: trials faster than 0.5 s (anticipated presses) are discarded, as well as the 63 windows of one subject whose recording is empty.

In [ ]:
plt.figure(figsize=(8, 3.3))
plt.hist(y, bins=40, color="lightsteelblue")
plt.axvspan(0.5, 2.5, color="orange", alpha=0.15, label="EEG window (0.5 s to 2.5 s after the stimulus)")
plt.xlabel("reaction time (s)"); plt.ylabel("trials")
plt.legend(); plt.show()

## 3. The subject explains about 20% of the variance

Left: the reaction time of each subject (sorted by median): subjects are not at the same level. Right: each subject's mean reaction time by age.

In [ ]:
df = pd.DataFrame({"rt": y, "subject": groups})
order = df.groupby("subject")["rt"].median().sort_values().index
data_box = [df.loc[df["subject"] == s, "rt"].to_numpy() for s in order]

rt_subject = (pd.DataFrame({"subject": groups, "rt": y, "release": releases})
                .groupby("subject").agg(mean_rt=("rt", "mean"), release=("release", "first")).reset_index())
aff = rt_subject.merge(participants[["participant_id", "age"]],
                       left_on="subject", right_on="participant_id")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].boxplot(data_box, showfliers=False)
ax[0].set_xticks([]); ax[0].set_xlabel("99 subjects (sorted by median)"); ax[0].set_ylabel("reaction time (s)")
for rel, g in aff.groupby("release"):
    ax[1].scatter(g["age"], g["mean_rt"], label=rel[:2], s=25)
ax[1].set_xlabel("age (years)"); ax[1].set_ylabel("subject's mean reaction time (s)")
ax[1].legend(title="release")
plt.tight_layout(); plt.show()

# share of variance explained by the subject: remove each subject's mean
y_c = y - pd.Series(y).groupby(groups).transform("mean").to_numpy()
print(f"share of variance explained by the subject: {1 - y_c.var() / y.var():.0%}")
print("correlation age / reaction time (99 subjects):", round(aff[["mean_rt", "age"]].corr().iloc[0, 1], 2))
for rel, g in aff.groupby("release"):
    print(rel[:2], round(g["age"].corr(g["mean_rt"]), 2))

- The subject explains about **20%** of the variance. The remaining **80%** are trial-to-trial variations within the same person: this is the part EEG should explain.
- Age explains little: correlation of **−0.31** over 99 subjects, but from −0.85 to +0.35 depending on the 20-subject release. With 20 subjects, this number is very unstable.
- **Consequence**: everything is evaluated with a per-subject `GroupKFold`, so that the model cannot recognise the person.

## 4. The references to beat: the mean and age alone

Before testing an EEG model, we measure two very simple models with a 5-fold `GroupKFold`:
- always predict the **mean** training reaction time;
- predict from **age alone** (linear regression).

Three small functions are used throughout the notebook: `rmse_folds` (RMSE per fold), `ridge_pipe` (imputation, standardisation, Ridge) and `alpha_sweep` (tries several `alpha`).

In [ ]:
gkf = GroupKFold(n_splits=5)

def rmse_folds(model, X, target):
    return -cross_val_score(model, X, target, groups=groups, cv=gkf,
                            scoring="neg_root_mean_squared_error")

def ridge_pipe(alpha):
    return Pipeline([("imputer", SimpleImputer(strategy="mean")),
                     ("scaler", StandardScaler()),
                     ("reg", Ridge(alpha=alpha))])

def alpha_sweep(X, target, alphas):
    rows = []
    for alpha in alphas:
        s = rmse_folds(ridge_pipe(alpha), X, target)
        rows.append({"alpha": alpha, "rmse_mean": s.mean(),
                     "rmse_worst": s.max(), "rmse_best": s.min()})
    return pd.DataFrame(rows).round(4)

res_raw, res_centered = {}, {}          # results kept for the summary plots

age_by_subject = participants.set_index("participant_id")["age"]
age_win = pd.Series(groups).map(age_by_subject).to_numpy().reshape(-1, 1)

s_mean = rmse_folds(DummyRegressor(), age_win, y)
s_age = rmse_folds(LinearRegression(), age_win, y)
print("mean      :", s_mean.mean().round(4), s_mean.round(3))
print("age alone :", s_age.mean().round(4), s_age.round(3))
res_raw["Mean (reference)"] = s_mean.mean()
res_raw["Age alone (reference)"] = s_age.mean()

The references are **0.3431** (mean) and **0.3355** (age alone). Age brings a small gain, about 2%. It is not an allowed input in the challenge: the "honest" reference is the mean.

The spread between folds is about ±0.015: a difference of 0.001 to 0.003 between two models is noise.

## 5. Classical models do not beat the mean

### 5.1 First family of features: power per frequency band

Each window is summarised by its power in 5 frequency bands (delta 1-4 Hz, theta 4-8, alpha 8-13, beta 13-30, gamma 30-50), computed for each channel with `scipy.signal.welch`. We take `log10` to compress the scale. Channel 128, the reference, is removed, leaving **640 variables** per trial. Flat channels (zero or near-zero power) become NaN, later replaced by the column mean.

In [ ]:
def band_power(window):
    f, psd = welch(window, fs=100, nperseg=100, axis=-1)
    bands = {"delta": (f >= 1) & (f < 4), "theta": (f >= 4) & (f < 8), "alpha": (f >= 8) & (f < 13),
             "beta": (f >= 13) & (f < 30), "gamma": (f >= 30) & (f <= 50)}
    return np.stack([psd[:, m].mean(axis=1) for m in bands.values()], axis=1).ravel()

power = np.stack([band_power(w) for w in X_win])
power_640 = power[:, :640].copy()               # without channel 128 (reference)
power_640[power_640 == 0] = np.nan              # exactly flat channel
log_power = np.log10(power_640)
log_power[log_power < -20] = np.nan             # nearly flat channel
print(log_power.shape, np.isnan(log_power).sum(), "NaN")

### 5.2 Ridge (regularised linear model)

We test several regularisation strengths `alpha`. The larger `alpha`, the more cautious the model.

In [ ]:
tab = alpha_sweep(log_power, y, [1e2, 1e3, 1e4, 1e5, 1e6])
print(tab)
res_raw["Ridge, band power"] = tab["rmse_mean"].min()

The score improves as `alpha` increases and stops at the level of the mean (0.3431): Ridge prefers to ignore the features.

### 5.3 XGBoost (decision trees, nonlinear)

In [ ]:
model = XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.5, random_state=0)
s = rmse_folds(model, log_power, y)
print(s.mean().round(4), s.round(3))
res_raw["XGBoost, band power"] = s.mean()

XGBoost does no better: a nonlinear model finds nothing either in the average power over 2 s.

### 5.4 Second family: the signal over time

Power over 2 s erases *when* things happen. We therefore keep the **mean of the signal per 200 ms slice**, for each of the 128 channels: 128 × 10 = **1,280 variables**.

The figure below compares the two families on the same signal (one channel of one trial). On the left, the "time" features: 10 means of 200 ms. On the right, the "power" features: the mean power in 5 frequency bands. The first keeps the *moment* but not the *rhythm*; the second keeps the *rhythm* but not the *moment*.

In [ ]:
i, c = 0, 50                       # trial and channel shown
sig = X_win[i, c]
t = np.arange(200) / 100           # time in seconds

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

# left: "time" features = 10 means of 200 ms
slice_mean = sig.reshape(10, 20).mean(axis=1)
ax[0].plot(t, sig, color="gray", lw=1)
for k in range(10):
    ax[0].axvline(k * 0.2, color="k", lw=0.3)
    ax[0].hlines(slice_mean[k], k * 0.2, (k + 1) * 0.2, colors="C3", lw=4)
ax[0].set(title="Time features: 10 means of 200 ms\n→ tell WHEN the signal rises or falls",
          xlabel="time after the start of the window (s)", ylabel="amplitude")

# right: "power" features = 5 means, one per frequency band
freqs, psd = welch(sig, fs=100, nperseg=100)
bands_plot = {"delta": (1, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 50.5)}
ax[1].semilogy(freqs, psd, color="gray")
for (name, (lo, hi)), col in zip(bands_plot.items(), ["C4", "C0", "C1", "C2", "C5"]):
    m = (freqs >= lo) & (freqs < hi)
    ax[1].fill_between(freqs[m], psd[m], psd[psd > 0].min(), color=col, alpha=0.3, label=name)
    ax[1].hlines(psd[m].mean(), lo, min(hi, 50), colors=col, lw=4)
ax[1].set(title="Power features: 5 means (one per band)\n→ tell WHICH frequencies dominate, but not when",
          xlabel="frequency (Hz)", ylabel="power")
ax[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
X_time = X_win[:, :128, :].reshape(-1, 128, 10, 20).mean(axis=3).reshape(-1, 128 * 10)
print(X_time.shape)
print(alpha_sweep(X_time, y, [1e2, 1e4, 1e6, 1e7]))

### 5.5 Removing the subject effect

We subtract each subject's mean from the features and from the reaction time: the model must then predict whether a trial is faster or slower than usual **for that person**. The reference becomes a model that always predicts 0.

In [ ]:
def center(X_, grp):
    X_df = pd.DataFrame(X_)
    return (X_df - X_df.groupby(grp).transform("mean")).to_numpy()

X_c = center(log_power, groups)           # band power, centred per subject
X_time_c = center(X_time, groups)         # 200 ms slices, centred per subject

s0 = rmse_folds(DummyRegressor(strategy="constant", constant=0.0), X_c, y_c)
print("zero (reference)       :", s0.mean().round(4))
res_centered["Zero (reference)"] = s0.mean()

t1 = alpha_sweep(X_c, y_c, [1e5])
print("Ridge, band power      :", t1["rmse_mean"].iloc[0])
res_centered["Ridge, band power"] = t1["rmse_mean"].iloc[0]

t2 = alpha_sweep(X_time_c, y_c, [1e4, 1e5, 1e6])
print(t2)
res_centered["Ridge, 200 ms slices"] = t2["rmse_mean"].min()

No model goes below the "predict 0" reference (0.3054). With small `alpha`, the time slices do much worse: one subject with large-amplitude artefacts is enough to throw the model off.

### 5.6 Neural network on the raw signal

I also trained **EEGNeX** (braindecode, PyTorch), the network from the starter kit, directly on the `(129, 200)` windows (notebook `eegnex.ipynb`). On one R5 fold, the validation RMSE stays between 0.34 and 0.38, against 0.335 for the mean, after 21 minutes of training.

### 5.7 Summary of the classical models

In [ ]:
def summary_plot(raw, centered):
    fig, ax = plt.subplots(1, 2, figsize=(13, 0.9 + 0.55 * max(len(raw), len(centered))))
    for a, title, d, ref in [(ax[0], "Target: reaction time", raw, raw["Mean (reference)"]),
                             (ax[1], "Subject effect removed", centered, centered["Zero (reference)"])]:
        names = list(d)[::-1]
        vals = [d[n] for n in names]
        a.scatter(vals, range(len(names)), s=80, zorder=3)
        a.axvline(ref, ls="--", color="gray")
        a.set_yticks(range(len(names))); a.set_yticklabels(names)
        for i, v in enumerate(vals):
            a.text(v + 0.0012, i, f"{v:.4f}", ha="left", va="center", fontsize=9)
        a.set_xlim(ref - 0.02, ref + 0.025)
        a.set_ylim(-0.6, len(names) - 0.4)
        a.set_title(title); a.set_xlabel("RMSE (s), dashed line = reference"); a.grid(axis="x", alpha=0.3)
    plt.tight_layout(); plt.show()

summary_plot(res_raw, res_centered)

No model goes clearly below its reference (the dashed line). The differences are smaller than the noise between folds.

## 6. Why: the variance of the features says nothing about reaction time

**PCA**: we look for the directions in which the 640 features vary the most, without looking at reaction time.

In [ ]:
X_std = StandardScaler().fit_transform(SimpleImputer(strategy="mean").fit_transform(log_power))
pca = PCA(n_components=10).fit(X_std)
Z = pca.transform(X_std)
print(pca.explained_variance_ratio_.round(3))

codes = pd.factorize(groups)[0]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab20", s=5)
ax[0].set_title("colour = subject")
sc = ax[1].scatter(Z[:, 0], Z[:, 1], c=y, cmap="viridis", s=5)
ax[1].set_title("colour = reaction time")
plt.colorbar(sc, ax=ax[1], label="s")
for a in ax:
    a.set_xlabel("component 1"); a.set_ylabel("component 2")
plt.tight_layout(); plt.show()

- The first two components explain **53%** of the variance. They describe a global power level shared by almost all channels, and a few very outlying windows from one or two subjects.
- Coloured by reaction time, the plot is mixed: no visible structure.

**Correlation with reaction time.** For each variable, we compute its correlation with reaction time, subject effect removed. We compare with a `y` shuffled within each subject.

In [ ]:
rng = np.random.default_rng(0)
y_perm = pd.Series(y_c).groupby(groups).transform(lambda s: rng.permutation(s.values)).to_numpy()

corr = pd.DataFrame(X_c).corrwith(pd.Series(y_c)).to_numpy()
corr_perm = pd.DataFrame(X_c).corrwith(pd.Series(y_perm)).to_numpy()
print("max |corr| real:", np.abs(corr).max().round(3), "| with shuffled y:", np.abs(corr_perm).max().round(3))

plt.figure(figsize=(6.5, 5))
plt.imshow(corr.reshape(128, 5), aspect="auto", cmap="coolwarm", vmin=-0.15, vmax=0.15)
plt.xticks(range(5), ["delta", "theta", "alpha", "beta", "gamma"])
plt.ylabel("channel"); plt.colorbar(label="correlation with reaction time")
plt.show()

chan19 = sorted([30, 35, 40, 41, 46, 54, 79, 92, 97, 102, 103, 105,    # motor (near C3, Cz, C4)
                 64, 69, 70, 74, 75, 82, 89])                          # occipital (near O1, Oz, O2)
others = [c for c in range(128) if c not in chan19]
cm = corr.reshape(128, 5)
print(pd.DataFrame({"19 motor+occipital channels": cm[chan19].mean(axis=0),
                    "other channels": cm[others].mean(axis=0)},
                   index=["delta", "theta", "alpha", "beta", "gamma"]).round(3))

- The largest correlation is **0.088**, against 0.033 with the shuffled `y`: the link exists, but it is tiny (less than 1% of the variance).
- It is **positive everywhere**: when power rises over the whole scalp, reaction time is slightly longer. This looks like a general state (movement, fatigue, attention), not a specific brain area.
- The 19 channels near the motor and visual cortex are no more correlated than the others.

## 7. What works a little: temporal splitting

A separate test (SVR) combines two initial ideas: keeping only channels close to the **motor cortex** (C3, Cz, C4) and the **visual cortex** (O1, Oz, O2), and splitting the window into **4 chunks of 500 ms**. The checks below show which of the two ideas really matters.

For each channel, band (theta 4-8 Hz, mu/alpha 8-13 Hz, beta 13-30 Hz) and chunk, we compute the `log10` of the power: 19 channels × 3 bands × 4 chunks = **228 variables**. The signal is first filtered in each band (`butter` + `filtfilt`). A flat channel on a chunk becomes a NaN.

Power is computed **for all 128 channels** once: choosing a subset of channels then just means indexing the array.

In [ ]:
BANDS = [(4, 8), (8, 13), (13, 30)]

def windowed_band_power(X, channels, bands, n_sub=4, sfreq=100, step=500):
    # output: (n_trials, n_channels, n_bands, n_sub), log10 of the power in µV²
    nyq = sfreq / 2
    filters = [butter(4, [lo / nyq, hi / nyq], btype="band") for lo, hi in bands]
    out = []
    for start in range(0, len(X), step):                      # in batches, to save memory
        Xc = X[start:start + step][:, channels, :].astype(np.float64) * 1e6      # µV
        n, n_ch, n_time = Xc.shape
        length = n_time // n_sub
        per_band = []
        for b, a in filters:
            f = filtfilt(b, a, Xc, axis=-1)[:, :, :length * n_sub]
            ms = (f ** 2).reshape(n, n_ch, n_sub, length).mean(axis=-1)
            p = np.log10(ms + 1e-12)
            p[ms < 1e-6] = np.nan                              # flat channel on this chunk
            per_band.append(p)
        out.append(np.stack(per_band, axis=2))
    return np.concatenate(out)

P = windowed_band_power(X_win, list(range(128)), BANDS)
F19 = P[:, chan19].reshape(len(P), -1)
print(P.shape, F19.shape)            # (6613, 128, 3, 4) (6613, 228)

The model is an **SVR with an rbf kernel**, heavily regularised (`C = 0.05`): it only moves away from the mean if the signal is clear. NaNs are replaced by the mean and variables are scaled with `RobustScaler` (insensitive to extreme values).

In [ ]:
def svr_pipe():
    return Pipeline([("imputer", SimpleImputer(strategy="mean")),
                     ("scaler", RobustScaler()),
                     ("svr", SVR(kernel="rbf", C=0.05, epsilon=0.05))])

s_svr = rmse_folds(svr_pipe(), F19, y)
print("SVR, 19 channels × 4 chunks :", s_svr.mean().round(4), s_svr.round(3))
print("mean (reference)            :", res_raw["Mean (reference)"].round(4))
res_raw["SVR, 19 channels × 4 chunks"] = s_svr.mean()

### Final test on a release never used

To avoid fooling ourselves after trying many settings, we keep **R5** (19 subjects) aside: the model is chosen and evaluated with a `GroupKFold` on R1 to R4, then tested **only once** on R5.

In [ ]:
test = releases == "R5_mini_L100_bdf"
dev = ~test

s_dev = -cross_val_score(svr_pipe(), F19[dev], y[dev], groups=groups[dev], cv=GroupKFold(n_splits=5),
                         scoring="neg_root_mean_squared_error")
s_dev_mean = -cross_val_score(DummyRegressor(), F19[dev], y[dev], groups=groups[dev], cv=GroupKFold(n_splits=5),
                              scoring="neg_root_mean_squared_error")
print("development R1 to R4: SVR", s_dev.mean().round(4), "| mean", s_dev_mean.mean().round(4))

pipe = svr_pipe().fit(F19[dev], y[dev])
rmse_test = np.sqrt(np.mean((pipe.predict(F19[test]) - y[test]) ** 2))
rmse_test_mean = np.sqrt(np.mean((y[test] - y[dev].mean()) ** 2))
print("final test R5       : SVR", rmse_test.round(4), "| mean", rmse_test_mean.round(4))

### Check 1: the permutation test

If we shuffle `y` at random, the SVR should no longer beat the mean. Otherwise, the "gain" comes from the method and not from the signal.

In [ ]:
rng = np.random.default_rng(0)
for i in range(3):
    s = rmse_folds(svr_pipe(), F19, rng.permutation(y))
    print("shuffled y", i + 1, ":", s.mean().round(4))
print("real y        :", s_svr.mean().round(4), "| mean:", res_raw["Mean (reference)"].round(4))

### Check 2: does the gain come from the choice of channels, or from temporal splitting?

We compare with three variants:
- **128 channels** × 4 chunks (1,536 variables);
- **19 randomly drawn channels** × 4 chunks, repeated 10 times (if the motor and occipital channels matter, they should do better than chance);
- **19 motor+occipital channels** with a **single window** of 2 s (no temporal splitting).

This cell is the longest of the notebook (a few minutes).

In [ ]:
s_all = rmse_folds(svr_pipe(), P.reshape(len(P), -1), y)
print("128 channels × 4 chunks        :", s_all.mean().round(4))

rng = np.random.default_rng(0)
draws = []
for i in range(10):
    idx = rng.choice(128, size=19, replace=False)
    draws.append(rmse_folds(svr_pipe(), P[:, idx].reshape(len(P), -1), y).mean())
print("19 random channels × 4         :", np.mean(draws).round(4), "±", np.std(draws).round(4),
      "| min", np.min(draws).round(4), "max", np.max(draws).round(4))
print("19 motor+occipital × 4         :", s_svr.mean().round(4))

P1 = windowed_band_power(X_win, chan19, BANDS, n_sub=1)
s_one = rmse_folds(svr_pipe(), P1.reshape(len(P1), -1), y)
print("19 motor+occipital, 1 window   :", s_one.mean().round(4))

**Reading.** Results of a separate series of checks (20 random draws, RMSE over the 5 releases). The outputs of this notebook should give close values.

| Variant | RMSE (5 releases) |
|---|---|
| Mean (reference) | 0.3431 |
| 19 motor+occipital channels × 4 chunks | 0.3371 |
| 128 channels × 4 chunks | 0.3411 |
| 19 random channels × 4 chunks (mean of 20 draws) | 0.3405 ± 0.0035 |
| 19 motor+occipital channels × 1 window of 2 s | 0.3449 |

- **Without temporal splitting, the gain disappears** (0.3449, no better than the mean).
- **With splitting, randomly drawn channels do almost as well**: 5 draws out of 20 equal or beat the motor+occipital selection. The "motor and visual cortex" choice is therefore not demonstrated: it is a favourable coincidence, not a property of the brain.
- The gain comes from the **temporal dynamics** of power, but it stays small: about 1% of the RMSE with random channels, about 2% for the best selection.

### Check 3: which chunk of the window carries the gain?

The window is anchored on the stimulus, but the button press happens at a variable moment (the reaction time). The late chunks of the window therefore contain the moment of the response. We test each chunk alone: if the gain comes mostly from the last ones, the model may be detecting **when the subject pressed** (movement, motor activity) rather than what happens before the response. This is a hypothesis, not a result.

In [ ]:
for k in range(4):
    Fk = P[:, chan19][:, :, :, k].reshape(len(P), -1)
    s = rmse_folds(svr_pipe(), Fk, y)
    print(f"chunk {k + 1} alone ({0.5 + 0.5 * k:.1f} s to {1.0 + 0.5 * k:.1f} s after the stimulus):", s.mean().round(4))
print("mean (reference)                                          :", res_raw["Mean (reference)"].round(4))

**How to read the output:** if a single chunk, especially one of the last ones, gives almost all of the gain, the hypothesis "the model spots the moment of the press" becomes plausible. If the four chunks are roughly equivalent, the signal is more diffuse. This check has not been interpreted yet.

## 8. Conclusion

In [ ]:
summary_plot(res_raw, res_centered)

- **No classical model beats the mean**, neither over the 99 subjects nor after removing the subject effect. Neither does a neural network.
- **A weak signal exists**: an SVR on power split into 4 chunks of 500 ms does better than the mean, by about 1 to 2%. It comes from the **temporal dynamics**, not from a specific brain area: randomly drawn channels do almost as well as the motor and visual channels. Without splitting, the gain disappears.
- This signal may reflect the moment of the response itself (the window contains the press) rather than the processing of the image: this is a hypothesis.
- The subject explains only about 20% of the variance: the rest is trial-to-trial variability, hard to read in 2 s of EEG.

**What I learned**
- With 20 subjects, everything is unstable (the age / reaction time correlation ranged from −0.85 to +0.35 depending on the release).
- References (mean, age alone) and a per-subject validation are needed before believing a score.
- Flat channels and near-zero values must be cleaned before standardising.
- A gain must be confirmed by checks: permutation test, held-out release, random channels. Here, the random-channel check corrected my initial interpretation ("motor and visual cortex").

**Next steps**: more subjects (R6 to R11), features centred on events (evoked potentials), better channel selection.